# GridMind SECOM benchmark companion

## tl;dr

Real public semiconductor measurements; logistic regression misses all 26 held-out failures and raises 7 false alarms at the frozen threshold. This is a reproducible negative experiment, not a deployable quality classifier or automotive/customer result.

## Context & Methods

The tested `benchmark.py` owns acquisition verification, preparation, modeling and metrics. This notebook calls it rather than duplicating the pipeline. Run the explicit `fetch.py` acquisition first.

### Key Assumptions

Source +1 means failed test. Timestamps are naive wall-clock test times; timezone, batch identity, feature semantics and actionable feature availability are not supplied. Earlier whole days train; later whole days test. No threshold/hyperparameter selection on the holdout. Statistical dependence can span days.

In [1]:
from pathlib import Path
import sys, json, csv, tempfile
from unittest.mock import patch
root = Path.cwd()
if not (root / "benchmark.py").exists():
    root = root / "benchmarks/secom"
sys.path.insert(0, str(root))
import benchmark as bench
X, y, times, source = bench.load_archive(root / "cache/secom.zip")
train, test = bench.chronological_split(times)
quality = bench.profile(X, y, times, train, test)

## Data

One source line is one production entity. All 590 actual numeric columns are anonymous measurements; labels/time are separate and excluded from X. UCI describes 591 features, a documented discrepancy. Source attribution: Michael McCann and Adrian Johnston (2008), UCI SECOM, DOI 10.24432/C54305, CC BY 4.0. GridMind derives splits, preprocessing, model and summaries; no endorsement.

In [2]:
print(json.dumps({"shape": quality["shape"], "labels": quality["labels"],
    "missing_cells": quality["missing_cells"], "missing_fraction": quality["missing_fraction"],
    "constant_columns": len(quality["nonempty_constant_columns"]),
    "duplicate_timestamps": quality["duplicate_timestamps"],
    "source_time_span": [quality["timestamp_min"], quality["timestamp_max"]],
    "train_rows": len(train), "test_rows": len(test),
    "archive_sha256": source["archive_sha256"]}, indent=2))

{
  "shape": [
    1567,
    590
  ],
  "labels": {
    "pass": 1463,
    "fail": 104,
    "rows": 1567,
    "failure_prevalence": 0.06636885768985322
  },
  "missing_cells": 41951,
  "missing_fraction": 0.04537548808583821,
  "constant_columns": 116,
  "duplicate_timestamps": 33,
  "source_time_span": [
    "2008-07-19T11:55:00",
    "2008-10-17T06:07:00"
  ],
  "train_rows": 1116,
  "test_rows": 451,
  "archive_sha256": "eea568baf3c2229096d7d294cf0b096b5502bd96d92c0b80a65b84714059be8e"
}


## Results

Both methods use identical rows. Median imputation, variance removal, scaling and balanced weights fit only training rows. This cell executes the complete offline experiment again and reconciles the exported predictions without the model scorer.

In [3]:
with tempfile.TemporaryDirectory() as directory, patch("urllib.request.urlopen", side_effect=AssertionError("Network forbidden in offline evaluation")):
    result = bench.run(root / "cache/secom.zip", Path(directory))
    with (Path(directory) / "held-out-predictions.csv").open() as stream:
        rows = list(csv.DictReader(stream))
    for name, field in [("majority", "majority_predicted_fail"), ("logistic_regression", "logistic_predicted_fail")]:
        cm = [[0, 0], [0, 0]]
        for row in rows:
            cm[int(row["actual_fail"])][int(row[field])] += 1
        assert cm == result["methods"][name]["confusion_matrix"]
print(json.dumps(result["methods"], indent=2))
print("Paired AP difference interval:", result["uncertainty"]["logistic_minus_majority"]["average_precision"])

{
  "majority": {
    "confusion_matrix": [
      [
        425,
        0
      ],
      [
        26,
        0
      ]
    ],
    "tn": 425,
    "fp": 0,
    "fn": 26,
    "tp": 0,
    "accuracy": 0.9423503325942351,
    "precision": null,
    "recall": 0.0,
    "specificity": 1.0,
    "balanced_accuracy": 0.5,
    "average_precision": 0.057649667405764965,
    "flagged_for_review": 0,
    "recall_wilson_95_iid_sensitivity": {
      "lower": 0.0,
      "upper": 0.1287289218592153,
      "successes": 0,
      "trials": 26,
      "assumption": "iid binomial sensitivity only; unknown production clustering may invalidate nominal coverage"
    },
    "precision_wilson_95_iid_sensitivity": null
  },
  "logistic_regression": {
    "confusion_matrix": [
      [
        418,
        7
      ],
      [
        26,
        0
      ]
    ],
    "tn": 418,
    "fp": 7,
    "fn": 26,
    "tp": 0,
    "accuracy": 0.926829268292683,
    "precision": 0.0,
    "recall": 0.0,
    "specificity": 0.9835

## Takeaways

The negative detection result is preserved. Average precision is 0.0678 versus a constant-score baseline 0.0576, with a paired day-bootstrap difference interval including zero. Zero-success bootstrap recall intervals are degenerate; the supplemental iid Wilson sensitivity is not a clustering-robust guarantee. Coefficients show associations among anonymous correlated measurements, not physical causes.

Next: document pre-outcome feature availability and group identities, define a consequential review decision/cost, and reserve a new future/group holdout before training-only tuning. No management action, savings, customer adoption or automotive transfer is established. Raw source data stay outside production Supabase and application APIs.